# Fraud Detection — Feature Engineering

## Objectives
The objective of this notebook is to engineer features motivated by `01_eda.ipynb` findings, decide how to handle `step`, and prepare the dataset for modeling.

**Feature engineering decisions:**

- `amount_to_oldbalanceOrg_ratio`
- `balanced_drained_flag`
- log-transform `amount` 
- encoding `type`
- dropping no signal and cosntant features
- deciding on keeping or dropping `step`

## Outputs

Cleaned, transformed, and scaled dataset saved to `data/processed/` as `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv`, ready for modeling in `03_modeling_and_evaluation.ipynb`.

## 2.1 Setup & Imports

Importing some of the same core libraries as `01_eda.ipynb`, with the addition of `os` for directory management, and sklearn module scaling. Path constants defined here to ensure reproducibility and consistent file references throughout the notebook. `os.makedirs()` is called to ensure the `data/processed/` directory exists before saving — preventing errors when running on a fresh clone of the repo. I split train and test by `step` because the model needs to be trained on historical data, and then deployed to catch fraud going forward, not using patterns learned from the future. Therefore, no sklearn import for train_test_split.  

In [1]:
import os

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

RAW_DATA_PATH = '../data/raw/paysim.csv'
X_TRAIN_PATH = '../data/processed/X_train.csv'
X_TEST_PATH = '../data/processed/X_test.csv'
Y_TRAIN_PATH = '../data/processed/y_train.csv'
Y_TEST_PATH = '../data/processed/y_test.csv'

os.makedirs('../data/processed/', exist_ok=True)

### Load Dataset

In [2]:
df = pd.read_csv(RAW_DATA_PATH)

## 2.2 EDA Finding → Feature Engineering Decisions

This section produces a table so feature engineering decisions and reasoning are organized in one area in an easily comprehensible format.

| EDA Finding | Feature | Hypothesis | Status |
|---|---|---|---|
| 0.86 fraud-specific correlation between `amount` and `oldbalanceOrg`, vs. ~0 for legit | `amount_to_oldbalanceOrg_ratio` | The relationship between amount and balance may separate fraud from legit more cleanly than either variable alone. Ratios near 1 indicate fraud. | Decided |
| `newbalanceOrig` clustering near zero in fraud | `balance_drained_ratio` (`newbalanceOrig / oldbalanceOrg`) | The clustering near zero in fraud makes the account-draining pattern an explicit feature rather than something the model has to infer from two raw columns. Ratios near 0 indicate draining. | Decided |
| `amount` mean and median were very different, suggesting skew/outliers | log-transform `amount` | Log transforming will compress the scale of large values, making relationships easier for the model to identify and allowing it to discriminate better. | Decided |
| Only `TRANSFER` and `CASH_OUT` had cases of fraud, out of 5 `type` categories | encode `type`, not restricted to the two fraud cases | Encoding all `type` categories ensures the model can be handed a real, live transaction of the other types and produce a sensible output. | Decided |
| No signal from `isFlaggedFraud` (all "No"), and no evidence found to support the chain relationship with `nameOrig`/`nameDest` | drop `isFlaggedFraud`, `nameOrig`, `nameDest` | They add no signaling value and would just be noise. | Decided |
| `step` was a very weak standalone signal for fraud | drop `step` as a model input feature | Not a strong standalone predictor; in correlation analysis it was always near 0, so it will likely only add noise. (Note: `step` is still used separately to define the train/test split cutoff — dropped as a feature, not dropped from the notebook entirely.) | Decided |

## 2.3 amount_to_oldbalanceOrg_ratio

Before engineering this feature, there is a risk of running into a division-by-zero error since `oldbalanceOrg==0` and `amount>0` could be a case (although extremely unlikely). I will filter out those instances in the dataset where  `oldbalanceOrg==0` and `amount>0` to decide whether I need to give this case a distinct binary flag as a fraud signal, or if its just a data-handling problem. 

In [3]:
# filter to isolate rows where oldbalanceOrg==0 and amount > 0
filtered_df = df[(df['oldbalanceOrg'] == 0) & (df['amount'] > 0)]
print(filtered_df['isFraud'].value_counts(normalize=True))
# print(filtered_df['isFraud'].mean())
filtered_df['type'].value_counts()

isFraud
0    0.999988
1    0.000012
Name: proportion, dtype: float64


type
CASH_OUT    1025783
PAYMENT      774245
TRANSFER     282783
CASH_IN       13464
DEBIT          6158
Name: count, dtype: int64

Since fraud is rarer in this specific subset (0.0012%) than the overall rate for either type (`TRANSFER`: 0.77%, `CASH_OUT`: 0.18%) that can even have fraud, this case is more likely a data-handling problem than a modeling signal, so I will skip the ratio calculation specifically for the rows in this case to avoid the divide-by-zero crash without discarding real data. 

To actually engineer this feature I use `np.where()` with the same conditional as use before for the filtered dataframe, and if the case is true then the value is set to `np.nan`, else it will be the ratio calculation. Because, `np.where()` calculates the ratio at every row, `np.where()` will compute `inf` for all the division-by-zero cases, immediately overwriting them with `np.nan`, but there will still be `RuntimeWarning`. To avoid this noise of `RuntimeWarning`s, I wrap the code in a `with np.errstate(divide='ignore')`. 

In [4]:
with np.errstate(divide='ignore'):
    df['amount_to_oldbalanceOrg_ratio'] = np.where((df['oldbalanceOrg']==0)&(df['amount']>0), np.nan, df['amount']/df['oldbalanceOrg'])

I use `df.head()` and filter specifically for rows I engineered the edge case for to check to make sure that the new feature was added correctly, and confirm the new column shows `NaN` at the edge cases.

In [5]:
print(df[(df['oldbalanceOrg']==0) & (df['amount']>0)][['amount','oldbalanceOrg','amount_to_oldbalanceOrg_ratio']].head())
df.head()

     amount  oldbalanceOrg  amount_to_oldbalanceOrg_ratio
29  9920.52            0.0                            NaN
30  3448.92            0.0                            NaN
31  4206.84            0.0                            NaN
32  5885.56            0.0                            NaN
33  5307.88            0.0                            NaN


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,amount_to_oldbalanceOrg_ratio
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0.057834
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0.087735
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,1.000000
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,1.000000
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0.280795


Rows 2 and 3 show evidence to support my reasoning of including this feature: 2 is `TRANSFER` with `amount`=181.00, `oldbalanceOrg` = 181.00, `isFraud` = 1, and the `amount_to_oldbalanceOrg_ratio` =  1.00, 3 is `CASH_OUT` with `amount`=181.00, `oldbalanceOrg` = 181.00, `isFraud` = 1, and the `amount_to_oldbalanceOrg_ratio` =  1.00. 

## 2.4 balance_drained_ratio

Before engineering this feature, there are two edge cases I need to consider handling. Case 1 if `oldbalanceOrg` and `newbalanceOrig` both equal 0, and case 2 if `oldbalanceOrg` = 0 but `newbalanceOrig` > 0. Depending on distribution and rarity, like before, I will decide on how to handle these edge cases.  

In [6]:
case1 = df[df['oldbalanceOrg']==0]
print(case1['newbalanceOrig'].value_counts())
case2 = df[(df['oldbalanceOrg'] == 0) & (df['newbalanceOrig'] > 0)]
print(case2['isFraud'].value_counts(normalize=True))
case2['type'].value_counts()

newbalanceOrig
0.00         2088985
133652.30          2
85553.24           2
99453.68           2
143236.26          1
              ...   
236649.24          1
304355.35          1
25645.63           1
17911.33           1
333125.32          1
Name: count, Length: 13462, dtype: int64
isFraud
0    1.0
Name: proportion, dtype: float64


type
CASH_IN    13464
Name: count, dtype: int64

Case 1 is overwhelmingly more common (2.09M cases) than case 2, only occurring in `CASH_IN` 13464 times. Given that case 1 (`0/0`, draining an empty account) is going to naturally resolve to `NaN` via numpy, I dont need any special case to handle this (likely will wrap in `with np.errstate(divide='ignore')` to avoid noise), and case 2 (nonzero newbalanceOrig from a 0 start) is 100% `CASH_IN`, 0% fraud, and not even a fraud capable type, therefore this pattern isn't a fraud signal. However, the ratio is numerically unsafe since it would compute `inf`, so it needs the same `np.nan` treatment as case 1 to keep the column clean for modeling. 

To actually engineer this feature I will use the same code as before, but with the else condition being `df['newbalanceOrig']/df['oldbalanceOrg']`. This will handle both cases since in case 1, numpy naturally handles it, and for case 2 the conditional will capture it and assign the value to `np.nan`.

In [7]:
with np.errstate(divide='ignore'):
    df['balance_drained_ratio'] = np.where((df['oldbalanceOrg']==0)&(df['newbalanceOrig']>0), np.nan, df['newbalanceOrig']/df['oldbalanceOrg'])

I use `df.head()` and filter specifically for rows I engineered the two edge case for to check to make sure that the new feature was added correctly, and confirm the new column shows `NaN` at the edge cases. I also use aggregates on the features grouped by `type` for `isFraud` to ensure the logic from earlier that all instances are `CASH_IN`.

In [8]:
print(df[(df['oldbalanceOrg']==0) & (df['newbalanceOrig']==0)][['oldbalanceOrg','newbalanceOrig', 'balance_drained_ratio']].head())
print(df[(df['oldbalanceOrg']==0) & (df['newbalanceOrig']>0)][['oldbalanceOrg','newbalanceOrig', 'balance_drained_ratio']].head())

df[(df['oldbalanceOrg']==0) & (df['newbalanceOrig']>0)].groupby('type')['isFraud'].agg(['sum', 'mean', 'count'])


    oldbalanceOrg  newbalanceOrig  balance_drained_ratio
29            0.0             0.0                    NaN
30            0.0             0.0                    NaN
31            0.0             0.0                    NaN
32            0.0             0.0                    NaN
33            0.0             0.0                    NaN
      oldbalanceOrg  newbalanceOrig  balance_drained_ratio
389             0.0       143236.26                    NaN
1260            0.0       378896.52                    NaN
3733            0.0        23669.02                    NaN
4308            0.0       208096.95                    NaN
5757            0.0       289490.08                    NaN


,sum,mean,count
type,,,
CASH_IN,0,0.0,13464


The output confirms the new column shows `NaN` at the correct edge cases. The `type` `count` confirms that the cases are 100% `CASH_IN` as expected. `sum` and `mean` of 0 confirms the fraud rate is 0%, which is expected since `CASH_IN` was a type incapable of holding fraud, found in EDA. 

## 2.5 Log-Transform `amount`

Because of the skew/outliers in `amount` and the large number zero-amount transactions, I will use `log1p` to transform `amount`. In downstream modeling, log transforming will compress the scale of large values, making relationships easier for the linear model to identify and discriminate. Random forest is invariant to this transformation, so it will neither harm nor help its performance. 

I will create a new column `amount_log`, so that I can keep the original dollar amounts. This will make it more readable for reference/debugging. 

In [9]:
df['amount_log'] = np.log1p(df['amount'])

In [13]:
# visually confrim the transformation and creation of the column was done correctly
print(df[['amount', 'amount_log']].head())
df[df['amount'] == 0][['amount', 'amount_log']].head()

     amount  amount_log
0   9839.64    9.194276
1   1864.28    7.531166
2    181.00    5.204007
3    181.00    5.204007
4  11668.14    9.364703


,amount,amount_log
2736447,0.0,0.0
3247298,0.0,0.0
3760289,0.0,0.0
5563714,0.0,0.0
5996408,0.0,0.0


The edge case log1p(0) = log(1) = 0 is correct, and the compression of `amount` to `amount_log` is visible and correct. 

## 2.6 Encode `type`

Given `type` has low cardinality and no natural order,  I will one-hot encode `type`. I include `drop_first=True` to avoid the dummy variable trap

In [15]:
df = pd.get_dummies(df, columns=['type'], drop_first=True)

In [16]:
# confirm columns added and row count is unchanged
print(df.columns.tolist())
print(df.shape)

df[[col for col in df.columns if col.startswith('type_')]].head()

['step', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'amount_to_oldbalanceOrg_ratio', 'balance_drained_ratio', 'amount_log', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']
(6362620, 17)


,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,False,False,True,False
1,False,False,True,False
2,False,False,False,True
3,True,False,False,False
4,False,False,True,False


Verified that one-hot encoding produced the expected columns (one per category except the dropped baseline), row count is unchanged, and spot-checked values show 1/0 patterns consistent with each row's original `type`. `CASH_IN` was dropped as the baseline category, meaning it's implicitly represented by all-`False` across the four `type_*` columns, and the linear model's coefficients for the other four types will be interpreted relative to `CASH_IN`.

## 2.7 Drop name columns

## 2.8 Train/Test Split By `step`

I will retain `step` for the split to avoid data leakage, preserving temporal order so the model isn't trained on future transactions to predict past ones, but I will drop it as a feature because `step` is a sequential index for this specific simulated dataset — `step` as a raw feature risks the model learning false associations tied to the dataset's specific timeline rather than any generalizable fraud pattern, which wouldn't transfer to new data.

## 2.9 Feature Scaling

## 2.10 Save Processed Data